<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/DjL_RIGOROUS_DATA_GENERATION_INTER_INTRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""
BLOCK 1 -- RIGOROUS DATA GENERATION
====================================================================
INTER (RR, RS) -> lowest-energy INTER conformer -> INTRA (full CREST)

    INTER
      RR  -> all actual structures, Q/theta/phi, classified
              (Boat / Twist-boat / Half-chair / ...)
      RS  -> all actual structures, Q/theta/phi, classified
    determine lowest-energy INTER conformer (per mother TS)
    INTRA
      lowest INTER conformer -> full CREST search
      retain EVERY returned conformer: energy, Q, theta, phi, raw xyz

Design rules followed here:
  - Every run starts ONLY from the mother-TS XYZ files in DRIVE_FOLDER.
    No step reads a CSV or folder left over from a previous run as an
    input (previous outputs are only ever regenerated, never assumed).
  - Every CREST call is wrapped in a tiered fallback: if a structure is
    difficult (times out / CREST errors / no conformers produced), the
    settings are progressively relaxed (thorough -> quick -> GFN-FF ->
    plain constrained optimization) rather than crashing the batch.
    If every tier fails, that one structure is logged and skipped; the
    rest of the batch continues.
  - Checkpointing: each stage (INTER per structure, INTRA per structure)
    is skipped if its output already exists and is complete, so a
    disconnect only costs you the structure that was in progress.
====================================================================
"""

import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 80)
print("BLOCK 1 SETUP")
print("=" * 80)

# ====================================================================
# 1. Install xtb + CREST
#    (apt + precompiled GitHub binary - fast, no conda/micromamba needed)
# ====================================================================

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "xtb"], check=True)

CREST_DIR = Path("/content/crest")
CREST_BIN = CREST_DIR / "crest"

if not CREST_BIN.exists():
    subprocess.run(
        ["wget", "-q", "-O", "/content/crest.tar.xz",
         "https://github.com/crest-lab/crest/releases/download/latest/"
         "crest-gnu-12-ubuntu-latest.tar.xz"],
        check=True,
    )
    CREST_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xf", "/content/crest.tar.xz", "-C", "/content"], check=True)
    CREST_BIN.chmod(0o755)

XTB_BIN = shutil.which("xtb") or "/usr/bin/xtb"
CREST = str(CREST_BIN)
XTB = str(XTB_BIN)

print("CREST:", CREST)
print("xTB:  ", XTB)
subprocess.run([CREST, "--version"], check=False)
subprocess.run([XTB, "--version"], check=False)

# ====================================================================
# 2. Mount Google Drive
# ====================================================================

from google.colab import drive

def mount_drive_with_retry(mountpoint="/content/drive", attempts=3):
    last_exc = None
    for i in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=True, timeout_ms=120000)
            return
        except Exception as exc:
            last_exc = exc
            print(f"Drive mount attempt {i}/{attempts} failed: {exc!r}")
            time.sleep(5)
    raise RuntimeError(
        "Could not mount Google Drive after several attempts.\n"
        "Most common fixes:\n"
        "  1. Allow third-party cookies for colab.research.google.com / accounts.google.com\n"
        "  2. Restart the runtime (Runtime -> Restart session) and run this cell first\n"
        "  3. Disable ad blockers / VPN for this session\n"
        "  4. Try mounting via the folder icon in the left sidebar instead\n"
    ) from last_exc

mount_drive_with_retry()

DRIVE_FOLDER = Path("/content/drive/MyDrive/Cycloetherification_xyz_files")
OUTPUT_ROOT = DRIVE_FOLDER / "BLOCK1_results"
INTER_ROOT = OUTPUT_ROOT / "INTER"
INTRA_ROOT = OUTPUT_ROOT / "INTRA"
for d in (OUTPUT_ROOT, INTER_ROOT, INTRA_ROOT):
    d.mkdir(parents=True, exist_ok=True)

if not DRIVE_FOLDER.exists():
    raise FileNotFoundError(f"Drive folder does not exist: {DRIVE_FOLDER}")

mother_files = sorted(DRIVE_FOLDER.glob("*.xyz"))
print(f"\nMother TS files found (fresh input only): {len(mother_files)}")
for p in mother_files:
    print(" ", p.name)

# ====================================================================
# 3. Configuration
#    Default ring/forming-bond numbering; per-file overrides supported.
# ====================================================================

DEFAULT_RING_ATOMS = [17, 18, 19, 20, 21, 22]   # O-C-C-C-C-C, ring connectivity order
DEFAULT_FORMING_BOND = [17, 22]

OVERRIDES = {
    # "SomeOtherNumbering.xyz": dict(ring_atoms=[1,2,3,4,5,6], forming_bond=[1,6]),
}

FC = 1.0                    # constraint force constant (Hartree/Bohr^2)
TORSION_TOLERANCE_DEG = 5.0        # INTER stage: collapses near-identical pucker families
INTRA_TOLERANCE_DEG = 5.0          # INTRA stage: set close to 0 (e.g. 0.1) to keep
                                    # essentially every CREST/CREGEN conformer individually
                                    # rather than merging near-duplicates at 5 deg
INTER_THREADS = 2
INTRA_THREADS = 2

# Tiered timeouts - restructured based on real observed behavior (see conversation):
# with only 2 threads, "thorough" GFN2-xTB reliably takes ~1h+ (CREST's own estimate)
# and is dropped entirely below. "Quick" GFN2-xTB is still often too slow (~14-24 min for
# a 6-MTD batch on 2 threads) so it gets a short, disposable attempt rather than the
# full 900s. GFN-FF is the proven fast/reliable workhorse (<1 min in practice) and is
# tried first now, not last.
GFN2_QUICK_TIMEOUT_SECONDS = 240   # short opportunistic attempt at real QM quality
GFNFF_TIMEOUT_SECONDS = 300        # generous; typically finishes in under a minute
PLAIN_OPT_TIMEOUT_SECONDS = 180    # last-resort single constrained optimization


def label_group(filename):
    """RR / RS / other, parsed from the filename prefix."""
    m = re.match(r"(RR|RS|SS|SR)", filename, flags=re.IGNORECASE)
    return m.group(1).upper() if m else "OTHER"


# ====================================================================
# 4. XYZ I/O
# ====================================================================

def read_xyz_frames(path):
    lines = Path(path).read_text().splitlines()
    frames = []
    pos = 0
    while pos < len(lines):
        if not lines[pos].strip():
            pos += 1
            continue
        n = int(lines[pos].strip())
        comment = lines[pos + 1] if pos + 1 < len(lines) else ""
        atom_lines = lines[pos + 2: pos + 2 + n]
        atoms, coords = [], []
        for line in atom_lines:
            parts = line.split()
            atoms.append(parts[0])
            coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
        frames.append({
            "atoms": atoms,
            "coords": np.asarray(coords, dtype=float),
            "comment": comment,
        })
        pos += n + 2
    return frames


def write_xyz_frames(frames, path):
    with open(path, "w") as f:
        for fr in frames:
            f.write(f"{len(fr['atoms'])}\n{fr.get('comment','')}\n")
            for el, xyz in zip(fr["atoms"], fr["coords"]):
                f.write(f"{el:2s} {xyz[0]: .10f} {xyz[1]: .10f} {xyz[2]: .10f}\n")


def parse_energy(comment):
    """Robust energy parse from a CREST/xtb xyz comment line."""
    m = re.search(r"([-+]?\d+\.\d+(?:[Ee][-+]?\d+)?)", comment)
    return float(m.group(1)) if m else np.nan


def dist(coords, i, j):
    return float(np.linalg.norm(coords[i - 1] - coords[j - 1]))


def dihedral(coords, i, j, k, l):
    p0, p1, p2, p3 = coords[i - 1], coords[j - 1], coords[k - 1], coords[l - 1]
    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2
    b1n = b1 / np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1n) * b1n
    w = b2 - np.dot(b2, b1n) * b1n
    return float(np.degrees(np.arctan2(np.dot(np.cross(b1n, v), w), np.dot(v, w))))


def ring_torsions(coords, ring):
    n = len(ring)
    return np.array([dihedral(coords, ring[k % n], ring[(k+1) % n],
                               ring[(k+2) % n], ring[(k+3) % n]) for k in range(n)])


def circular_diff(a, b):
    return np.abs((a - b + 180.0) % 360.0 - 180.0)


# ====================================================================
# 5. Cremer-Pople (N=6) + classification
# ====================================================================

def cremer_pople_6(ring_coords):
    R = np.asarray(ring_coords, dtype=float)
    Rc = R - R.mean(axis=0)
    _, _, vh = np.linalg.svd(Rc, full_matrices=False)
    normal = vh[-1]
    z = Rc @ normal
    j = np.arange(6)
    c = np.sqrt(2/6) * np.sum(z * np.cos(2*2*np.pi*j/6))
    s = -np.sqrt(2/6) * np.sum(z * np.sin(2*2*np.pi*j/6))
    q2 = np.sqrt(c**2 + s**2)
    phi = np.degrees(np.arctan2(s, c)) % 360.0
    q3 = np.sqrt(1/6) * np.sum(z * ((-1.0) ** j))
    Q = np.sqrt(q2**2 + q3**2)
    theta = np.degrees(np.arccos(np.clip(q3 / Q, -1, 1))) if Q > 1e-9 else 0.0
    return float(Q), float(theta), float(phi)


def classify_6ring(theta, phi):
    phi = phi % 360.0
    if theta < 15 or theta > 165:
        return "Chair (C)"
    if 75 <= theta <= 105:
        m = phi % 60
        return "Boat (B)" if (m < 15 or m > 45) else "Twist-boat (S)"
    if 35 <= theta <= 65 or 115 <= theta <= 145:
        m = phi % 60
        return "Envelope (E)" if (m < 15 or m > 45) else "Half-chair (H)"
    return "Intermediate"


# ====================================================================
# 6. Robust, tiered CREST runner
#    Tier 1: short GFN2-xTB "quick" attempt (real QM quality, abandoned fast
#            via GFN2_QUICK_TIMEOUT_SECONDS if it's not going to finish)
#    Tier 2: GFN-FF "quick" (classical force field - proven fast/reliable
#            workhorse; tried second, not last, based on observed behavior)
#    Tier 3: plain constrained xtb optimization (single structure only,
#            last-resort fallback, guarantees at least one usable result)
#    Every tier is also watched live: if CREST prints its own "Estimated
#    runtime for a batch of N MTDs..." line and that estimate already
#    exceeds the remaining timeout budget, the attempt is aborted
#    immediately rather than waiting out the full timeout.
#    "Thorough" GFN2-xTB is intentionally not attempted at all by default -
#    with 2 threads it realistically takes 1h+ for substrates this size.
# ====================================================================

def write_toml(path, xyz_name, method, threads, forming_bond, fc):
    path.write_text(f"""input = '{xyz_name}'
runtype = 'imtd-gc'
threads = {threads}

[[calculation.level]]
method = "{method}"

[[calculation.constraint]]
type = 'bond'
atoms = [{forming_bond[0]}, {forming_bond[1]}]
fc = {fc}
""")


_RUNTIME_ESTIMATE_RE = re.compile(
    r"Estimated runtime for a batch of \d+ MTDs on \d+ threads:\s*"
    r"(?:(\d+)\s*h)?\s*(?:(\d+)\s*min)?\s*(?:([\d.]+)\s*sec)?"
)


def _parse_estimated_seconds(line):
    """Parses CREST's own 'Estimated runtime for a batch of N MTDs on T
    threads: 1 h 4 min 29 sec' style line into total seconds, or None if the
    line doesn't match."""
    m = _RUNTIME_ESTIMATE_RE.search(line)
    if not m or not any(m.groups()):
        return None
    h, mn, s = m.groups()
    return (float(h or 0) * 3600) + (float(mn or 0) * 60) + float(s or 0)


def run_with_live_output(cmd, cwd, timeout_s, log_path, log=print, heartbeat_every=30,
                          abort_if_estimate_exceeds=True):
    """Runs cmd, streaming stdout to log_path AND to the console live, so a long
    CREST run shows visible progress instead of going silent until it finishes
    (or looking hung). Uses a background thread to read output so the timeout
    and heartbeat checks keep running even during stretches where the
    subprocess produces zero output (readline() alone would block forever
    and never let a timeout fire in that case).

    If abort_if_estimate_exceeds is True, this also watches for CREST's own
    "Estimated runtime for a batch of N MTDs on T threads: ..." line and, the
    moment it appears, checks whether that estimate already exceeds the
    remaining timeout budget. If so, it kills the run immediately instead of
    waiting out the full timeout for a result that's already predictable -
    this is the single biggest time-saver when a tier is reliably too slow
    for the available thread count (see conversation: Tier 1/2 were both
    burning the full 900s on every attempt, never once succeeding, before
    falling back to Tier 3 - this catches that in ~60-90s instead.

    Returns (returncode_or_None, timed_out: bool)."""
    import queue
    import threading

    cwd = Path(cwd)
    log_path = Path(log_path)
    start = time.time()
    last_heartbeat = start

    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )

    q = queue.Queue()

    def _reader():
        try:
            for line in iter(proc.stdout.readline, ""):
                q.put(line)
        finally:
            q.put(None)  # sentinel: stdout closed / process done producing output

    t = threading.Thread(target=_reader, daemon=True)
    t.start()

    stream_done = False
    with open(log_path, "w") as logfile:
        while True:
            now = time.time()
            if now - start > timeout_s:
                proc.kill()
                proc.wait()
                log(f"      [{now-start:6.0f}s] TIMEOUT - killed after {timeout_s}s")
                return None, True

            try:
                line = q.get(timeout=1.0)  # non-blocking-ish: re-checks timeout every 1s
            except queue.Empty:
                line = ""

            if line is None:
                stream_done = True
            elif line:
                logfile.write(line)
                logfile.flush()
                stripped = line.strip()
                if any(k in stripped for k in
                       ("MTD", "GC", "CREGEN", "error", "Error", "ERROR",
                        "warning", "unique conformers", "terminated")):
                    log(f"      [{time.time()-start:6.0f}s] {stripped}")

                if abort_if_estimate_exceeds:
                    est = _parse_estimated_seconds(stripped)
                    if est is not None:
                        remaining = timeout_s - (time.time() - start)
                        if est > remaining:
                            proc.kill()
                            proc.wait()
                            log(f"      [{time.time()-start:6.0f}s] CREST's own estimate "
                                f"({est:.0f}s) exceeds remaining timeout budget "
                                f"({remaining:.0f}s) - aborting early instead of waiting "
                                f"out the full timeout.")
                            return None, True

            if stream_done and proc.poll() is not None:
                return proc.returncode, False

            if not line and now - last_heartbeat > heartbeat_every:
                log(f"      ... still running ({now-start:6.0f}s elapsed, "
                    f"timeout at {timeout_s}s)")
                last_heartbeat = now


def try_crest_tier(xyz_file, workdir, forming_bond, fc, method, quick, threads, timeout_s, log=print):
    workdir = Path(workdir)
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(xyz_file, workdir / "struc.xyz")

    toml_path = workdir / "input.toml"
    write_toml(toml_path, "struc.xyz", method, threads, forming_bond, fc)

    cmd = [CREST, "--input", toml_path.name] + (["-mquick"] if quick else [])
    rc, timed_out = run_with_live_output(
        cmd, workdir, timeout_s, workdir / "crest_stdout.log", log=log,
    )
    if timed_out:
        return None, "timeout"

    conf_file = workdir / "crest_conformers.xyz"
    if rc != 0 or not conf_file.exists():
        return None, f"crest_failed (rc={rc})"

    frames = read_xyz_frames(conf_file)
    for fr in frames:
        fr["energy_Eh"] = parse_energy(fr["comment"])

    if not frames:
        return None, "no_conformers"

    return frames, "ok"


def try_plain_optimization_tier(xyz_file, workdir, forming_bond, fc, method, timeout_s, log=print):
    """Last-resort fallback: single constrained xtb geometry optimization,
    no conformer search. Guarantees one usable structure if CREST itself
    keeps failing for this molecule."""
    workdir = Path(workdir)
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(xyz_file, workdir / "struc.xyz")

    constraint_path = workdir / "xtb_constraint.inp"
    constraint_path.write_text(
        f"$constrain\n"
        f"distance: {forming_bond[0]}, {forming_bond[1]}, auto\n"
        f"force constant={fc}\n"
        f"$end\n"
    )

    gfn_flag = "--gfn2" if method == "gfn2" else "--gfnff"
    cmd = [XTB, "struc.xyz", "--opt", gfn_flag, "--input", constraint_path.name]
    rc, timed_out = run_with_live_output(
        cmd, workdir, timeout_s, workdir / "xtb_stdout.log", log=log,
    )
    if timed_out:
        return None, "timeout"

    opt_file = workdir / "xtbopt.xyz"
    if rc != 0 or not opt_file.exists():
        return None, f"xtb_opt_failed (rc={rc})"

    frames = read_xyz_frames(opt_file)
    if not frames:
        return None, "no_output_structure"

    m = re.search(r"energy:\s*([-+]?\d+\.\d+)", frames[0]["comment"])
    frames[0]["energy_Eh"] = float(m.group(1)) if m else parse_energy(frames[0]["comment"])
    return frames, "ok (plain optimization fallback)"


def robust_crest_search(xyz_file, workdir_base, forming_bond, fc,
                         preferred_method="gfn2", threads=2, log=print):
    """Runs the tiered fallback. Returns (frames, tier_used) or (None, reason).

    Tier 1: short GFN2-xTB quick attempt (real QM quality, abandoned fast if
            it's not going to finish - see GFN2_QUICK_TIMEOUT_SECONDS)
    Tier 2: GFN-FF quick (fast, classical-force-field workhorse - proven
            reliable for this substrate size at 2 threads)
    Tier 3: plain constrained optimization (single structure, last resort)

    "Thorough" GFN2-xTB is intentionally NOT attempted by default: with 2
    threads it realistically takes 1h+ for substrates this size (confirmed by
    CREST's own runtime estimate in testing), so it never actually pays off
    before falling back anyway - skipping it saves the wasted wait entirely.
    """
    tiers = [
        dict(method=preferred_method, quick=True, timeout=GFN2_QUICK_TIMEOUT_SECONDS,
             label="Tier 1: quick GFN2 (short attempt)"),
        dict(method="gff", quick=True, timeout=GFNFF_TIMEOUT_SECONDS,
             label="Tier 2: GFN-FF quick (primary workhorse)"),
    ]
    for i, tier in enumerate(tiers, start=1):
        log(f"    {tier['label']} (timeout {tier['timeout']}s) ...")
        frames, status = try_crest_tier(
            xyz_file, f"{workdir_base}_t{i}", forming_bond, fc,
            tier["method"], tier["quick"], threads, tier["timeout"], log=log,
        )
        if frames:
            log(f"    -> succeeded ({tier['label']}, {len(frames)} conformers)")
            return frames, tier["label"]
        log(f"    -> failed ({status}), falling back...")

    log(f"    Tier 3: plain constrained optimization (last resort, "
        f"timeout {PLAIN_OPT_TIMEOUT_SECONDS}s) ...")
    frames, status = try_plain_optimization_tier(
        xyz_file, f"{workdir_base}_t3", forming_bond, fc, preferred_method,
        PLAIN_OPT_TIMEOUT_SECONDS, log=log,
    )
    if frames:
        log(f"    -> succeeded (Tier 3, 1 structure)")
        return frames, "Tier 3: plain optimization"

    log(f"    -> ALL TIERS FAILED ({status}). Skipping this structure.")
    return None, status


# ====================================================================
# 7. Torsion-based deduplication (5 deg, all 6 ring torsions)
# ====================================================================

def dedupe_by_torsions(frames, ring_atoms, tolerance_deg=TORSION_TOLERANCE_DEG):
    indexed = [(i, fr) for i, fr in enumerate(frames)]
    indexed.sort(key=lambda t: (not np.isfinite(t[1].get("energy_Eh", np.nan)),
                                 t[1].get("energy_Eh", np.inf)))
    kept = []
    for orig_idx, fr in indexed:
        tors = ring_torsions(fr["coords"], ring_atoms)
        dup_of = None
        for k in kept:
            if np.all(circular_diff(tors, k["torsions"]) <= tolerance_deg):
                dup_of = k["retained_id"]
                break
        if dup_of is None:
            kept.append({"frame": fr, "torsions": tors,
                         "retained_id": len(kept) + 1, "orig_idx": orig_idx})
    return kept


def build_rows(kept_list, ring_atoms, forming_bond, tag):
    rows = []
    energies = [k["frame"]["energy_Eh"] for k in kept_list]
    emin = np.nanmin(energies) if len(energies) else np.nan
    for k in kept_list:
        fr = k["frame"]
        Q, theta, phi = cremer_pople_6(fr["coords"][[a - 1 for a in ring_atoms]])
        e = fr["energy_Eh"]
        rows.append({
            "structure": tag,
            "conformer": k["retained_id"],
            "energy_Eh": e,
            "rel_kcal": (e - emin) * 627.5095 if np.isfinite(e) and np.isfinite(emin) else np.nan,
            "forming_bond_A": dist(fr["coords"], forming_bond[0], forming_bond[1]),
            "Q_A": Q, "theta_deg": theta, "phi_deg": phi,
            "classification": classify_6ring(theta, phi),
        })
    return pd.DataFrame(rows).sort_values("rel_kcal").reset_index(drop=True)


def save_conformers(kept_list, df, out_dir, tag, prefix):
    out_dir.mkdir(parents=True, exist_ok=True)
    for k, (_, row) in zip(kept_list, df.iterrows()):
        cls_tag = row["classification"].split(" ")[0].replace("/", "-")
        fname = f"{tag}_{prefix}{int(row['conformer']):03d}_rel{row['rel_kcal']:.3f}kcal_{cls_tag}.xyz"
        fr = dict(k["frame"])
        fr["comment"] = (f"{prefix} conformer={row['conformer']} energy_Eh={row['energy_Eh']:.8f} "
                          f"rel_kcal={row['rel_kcal']:.4f} class={row['classification']}")
        write_xyz_frames([fr], out_dir / fname)
    df.to_csv(out_dir / f"{tag}_{prefix}_results.csv", index=False)


# ====================================================================
# 8. INTER stage - per mother TS, grouped by RR/RS
# ====================================================================

def inter_is_complete(out_dir, tag):
    return (out_dir / f"{tag}_INTER_results.csv").exists()


def run_inter_stage(xyz_file, ring_atoms, forming_bond, log=print):
    tag = xyz_file.stem
    group = label_group(xyz_file.name)
    out_dir = INTER_ROOT / group / tag

    if inter_is_complete(out_dir, tag):
        log(f"  INTER already complete for {tag}, loading saved results.")
        return pd.read_csv(out_dir / f"{tag}_INTER_results.csv"), out_dir

    frames, tier_used = robust_crest_search(
        xyz_file, f"/content/inter_{tag}", forming_bond, FC,
        preferred_method="gfn2", threads=INTER_THREADS, log=log,
    )
    if frames is None:
        return None, out_dir

    kept = dedupe_by_torsions(frames, ring_atoms)
    df = build_rows(kept, ring_atoms, forming_bond, tag)
    df["tier_used"] = tier_used
    save_conformers(kept, df, out_dir, tag, "INTER")
    return df, out_dir


# ====================================================================
# 9. INTRA stage - full search seeded from the lowest INTER conformer
# ====================================================================

def intra_is_complete(out_dir, tag):
    return (out_dir / f"{tag}_INTRA_results.csv").exists()


def run_intra_stage(tag, seed_xyz_path, ring_atoms, forming_bond, log=print):
    out_dir = INTRA_ROOT / tag

    if intra_is_complete(out_dir, tag):
        log(f"  INTRA already complete for {tag}, loading saved results.")
        return pd.read_csv(out_dir / f"{tag}_INTRA_results.csv")

    frames, tier_used = robust_crest_search(
        seed_xyz_path, f"/content/intra_{tag}", forming_bond, FC,
        preferred_method="gfn2", threads=INTRA_THREADS, log=log,
    )
    if frames is None:
        return None

    # INTRA retains every distinct conformer CREST/CREGEN returns; INTRA_TOLERANCE_DEG
    # controls how aggressively near-duplicates are merged (set it near 0 to keep
    # essentially everything CREST itself didn't already collapse).
    kept = dedupe_by_torsions(frames, ring_atoms, tolerance_deg=INTRA_TOLERANCE_DEG)
    df = build_rows(kept, ring_atoms, forming_bond, tag)
    df["tier_used"] = tier_used
    save_conformers(kept, df, out_dir, tag, "INTRA")
    return df


# ====================================================================
# 10. RUN BLOCK 1 over every mother TS
# ====================================================================

all_inter_rows = []
all_intra_rows = []
failures = []
progress_path = OUTPUT_ROOT / "BLOCK1_progress.csv"
failure_path = OUTPUT_ROOT / "BLOCK1_failures.csv"

for n, xyz_file in enumerate(mother_files, start=1):
    tag = xyz_file.stem
    cfg = dict(ring_atoms=DEFAULT_RING_ATOMS, forming_bond=DEFAULT_FORMING_BOND)
    cfg.update(OVERRIDES.get(xyz_file.name, {}))

    print()
    print("=" * 80)
    print(f"[{n}/{len(mother_files)}] {xyz_file.name}  (group={label_group(xyz_file.name)})")
    print("=" * 80)

    try:
        print("  -- INTER stage --")
        inter_df, inter_dir = run_inter_stage(xyz_file, cfg["ring_atoms"], cfg["forming_bond"])
        if inter_df is None or len(inter_df) == 0:
            raise RuntimeError("INTER stage produced no usable conformers")

        inter_tagged = inter_df.copy()
        inter_tagged.insert(0, "group", label_group(xyz_file.name))
        all_inter_rows.append(inter_tagged)

        lowest_row = inter_df.iloc[0]
        cls_tag = str(lowest_row["classification"]).split(" ")[0].replace("/", "-")
        seed_xyz = inter_dir / (f"{tag}_INTER{int(lowest_row['conformer']):03d}_"
                                 f"rel{lowest_row['rel_kcal']:.3f}kcal_{cls_tag}.xyz")
        if not seed_xyz.exists():
            candidates = list(inter_dir.glob(f"{tag}_INTER{int(lowest_row['conformer']):03d}_*.xyz"))
            if not candidates:
                raise FileNotFoundError(f"Could not locate seed xyz for lowest INTER conformer of {tag}")
            seed_xyz = candidates[0]

        print(f"  Lowest-energy INTER conformer: #{int(lowest_row['conformer'])} "
              f"({lowest_row['classification']}, 0.000 kcal/mol) -> seeding INTRA")

        print("  -- INTRA stage --")
        intra_df = run_intra_stage(tag, seed_xyz, cfg["ring_atoms"], cfg["forming_bond"])
        if intra_df is None or len(intra_df) == 0:
            raise RuntimeError("INTRA stage produced no usable conformers")

        intra_tagged = intra_df.copy()
        intra_tagged.insert(0, "group", label_group(xyz_file.name))
        all_intra_rows.append(intra_tagged)

        print(f"  Completed {tag}: INTER={len(inter_df)} retained, INTRA={len(intra_df)} retained")

    except Exception as exc:
        print(f"  FAILED: {tag} -> {exc!r}")
        failures.append({"structure": tag, "error": repr(exc)})

    # checkpoint after every structure
    if all_inter_rows:
        pd.concat(all_inter_rows, ignore_index=True).to_csv(
            OUTPUT_ROOT / "ALL_INTER_results.csv", index=False)
    if all_intra_rows:
        pd.concat(all_intra_rows, ignore_index=True).to_csv(
            OUTPUT_ROOT / "ALL_INTRA_results.csv", index=False)
    pd.DataFrame(failures).to_csv(failure_path, index=False)

print()
print("=" * 80)
print("BLOCK 1 FINISHED")
print("=" * 80)
print(f"Structures processed: {len(mother_files)}")
print(f"Failures: {len(failures)}")
print(f"Results folder: {OUTPUT_ROOT}")
if failures:
    print("\nFailed structures (see BLOCK1_failures.csv):")
    for f in failures:
        print(" ", f["structure"], "-", f["error"])

BLOCK 1 SETUP
CREST: /content/crest/crest
xTB:   /usr/bin/xtb
Mounted at /content/drive

Mother TS files found (fresh input only): 34
  RR_CH2-Ph.xyz
  RR_CH2-dioxane.xyz
  RR_CH2CN.xyz
  RR_CH2CO2Me.xyz
  RR_CMe2CO2Me.xyz
  RR_Cy.xyz
  RR_Et.xyz
  RR_Me-Propane.xyz
  RR_Me.xyz
  RR_Ph.xyz
  RR_alkene.xyz
  RR_alkyne.xyz
  RR_allyl.xyz
  RR_dimethyl-allyl.xyz
  RR_iPr.xyz
  RR_nBu.xyz
  RR_tBu.xyz
  RS_CH2-Ph.xyz
  RS_CH2-dioxane.xyz
  RS_CH2CN.xyz
  RS_CH2CO2Me.xyz
  RS_CMe2CO2Me.xyz
  RS_Cy.xyz
  RS_Et.xyz
  RS_Me-Propane.xyz
  RS_Me.xyz
  RS_Ph.xyz
  RS_alkene.xyz
  RS_alkyne.xyz
  RS_allyl.xyz
  RS_dimethyl-allyl.xyz
  RS_iPr.xyz
  RS_nBu.xyz
  RS_tBu.xyz

[1/34] RR_CH2-Ph.xyz  (group=RR)
  -- INTER stage --
    Tier 1: quick GFN2 (short attempt) (timeout 240s) ...
      [     0s] -mquick  : very crude quick-mode (no NORMMD, no GC, crude opt.)
      [     2s] │              CREST iMTD-GC SAMPLING             │
      [     2s] Generating MTD length from a flexibility measure
      [